# Competition Submission

End-to-end Solafune tree canopy pipeline aimed at **weighted instance mAP**:

- DeepLabV3+ (ResNet-34, ImageNet) on RGB
- Weighted CE for class imbalance
- Distance-transform + watershed instance split before polygon export
- Submission JSON matching `sample_answer.json`

Requires `02` / `03` completed (`data/train_images`, masks). Hyperparameters from `config.yaml`.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.annotations import load_json_annotations
from src.data.augmentations import get_train_augmentations, get_val_augmentations
from src.data.image_loader import load_image
from src.data.loaders import ImageMaskDataset
from src.exploration.class_explorer import color_mask, plot_training_history
from src.exploration.evaluation import load_best_model
from src.exploration.visualize import show_side_by_side
from src.prediction.pipeline import Predictor
from src.prediction.submission import (
    export_submission,
    mask_to_polygons_multiclass,
    separate_instances,
)
from src.training.engine import run_training
from src.training.metrics import compute_metrics_multiclass
from src.training.running import get_version_config
from src.training.trainer import create_splits
from src.utils.config import Config
from src.utils.helpers import c, init_notebook, p, t
from src.utils.versioning import VersionManager

config = Config.load(root=root)
init_notebook(config.train.seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

NOTEBOOK_ID = "16"
MODEL_NAME = "smp_deeplabv3plus"
ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"
MODE = "rgb"
IN_CHANNELS = 3
MODEL_KWARGS = {"encoder_name": ENCODER_NAME, "encoder_weights": ENCODER_WEIGHTS}

p("Device", device)
p("Image size", config.train.image_size)
p("Batch / epochs / lr", f"{config.train.batch_size} / {config.train.epochs} / {config.train.learning_rate}")
p("Model", f"{MODEL_NAME} + {ENCODER_NAME}")


## Data + ground-truth samples


In [ ]:
train_dir = config.paths.train_images
entries = load_json_annotations(config.paths.annotations)
train_entries, val_entries = create_splits(entries)

train_tf = get_train_augmentations(config.train.image_size, mode=MODE)
val_tf = get_val_augmentations(config.train.image_size, mode=MODE)

train_ds = ImageMaskDataset(train_entries, train_dir, transform=train_tf)
val_ds = ImageMaskDataset(val_entries, train_dir, transform=val_tf)

pin = device.type == "cuda"
train_loader = DataLoader(
    train_ds,
    batch_size=config.train.batch_size,
    shuffle=True,
    num_workers=config.train.num_workers,
    pin_memory=pin,
)
val_loader = DataLoader(
    val_ds,
    batch_size=config.train.batch_size,
    shuffle=False,
    num_workers=config.train.num_workers,
    pin_memory=pin,
)

p("Train / val", f"{len(train_ds)} / {len(val_ds)}")

t("Ground-truth samples")
for entry in train_entries[:3]:
    img = load_image(train_dir, entry)
    gt = color_mask(config, entry)
    show_side_by_side(
        img,
        gt,
        titles=(entry.image_path.name, "GT polygons"),
        mask_colors=config.MASK_COLORS,
    )


## Train


In [ ]:
key, version_root, exp_config, best_model_path, checkpoint_path_check, best_model_exists = get_version_config(
    config,
    None,
    NOTEBOOK_ID,
    MODEL_NAME,
    MODE,
    str(IN_CHANNELS),
    None,
    None,
    None,
)

p("Version root", version_root)
p("Existing best", best_model_exists)

trainer = None
if best_model_exists:
    p("Reusing checkpoint", best_model_path, color1=c.ORANGE)
else:
    t("Training")
    trainer = run_training(
        config=exp_config,
        train_loader=train_loader,
        val_loader=val_loader,
        version_root=version_root,
        model_name=MODEL_NAME,
        in_channels=IN_CHANNELS,
        **MODEL_KWARGS,
    )


## Metric plots


In [ ]:
if trainer is not None:
    plot_training_history(trainer, title_prefix="Competition model")

    hist = trainer.history
    epochs = range(1, len(hist.get("train_loss", [])) + 1)
    if epochs:
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

        ax = axes[0]
        ax.plot(epochs, hist["train_loss"], label="train")
        ax.plot(epochs, hist["val_loss"], label="val")
        ax.set_title("Loss")
        ax.set_xlabel("Epoch")
        ax.legend()

        ax = axes[1]
        ax.plot(epochs, hist["val_iou"], label="mean IoU")
        ax.plot(epochs, hist["val_f1"], label="F1")
        ax.set_title("IoU / F1")
        ax.set_xlabel("Epoch")
        ax.legend()

        ax = axes[2]
        ax.plot(epochs, hist["val_precision"], label="precision")
        ax.plot(epochs, hist["val_recall"], label="recall")
        ax.set_title("Precision / Recall")
        ax.set_xlabel("Epoch")
        ax.legend()

        fig.suptitle("Validation curves")
        plt.tight_layout()
        plt.show()
else:
    p("No new trainer history", "skipped epoch plots", color1=c.ORANGE)


## Load best weights + validation samples


In [ ]:
model_dir = (
    config.paths.models
    / NOTEBOOK_ID
    / MODEL_NAME
    / MODE
    / f"size_{config.train.image_size}"
)
vm = VersionManager(model_dir)
version_folder = vm.find_latest()
if version_folder is None:
    raise RuntimeError(f"No version under {model_dir}")

model = load_best_model(
    MODEL_NAME,
    config,
    notebook=NOTEBOOK_ID,
    mode=MODE,
    **MODEL_KWARGS,
)

temp_path = config.paths.models / "temp_competition_model.pth"
torch.save({"model": model.state_dict()}, temp_path)

predictor = Predictor(
    model_path=temp_path,
    model_name=MODEL_NAME,
    image_size=config.train.image_size,
    **MODEL_KWARGS,
)

totals = {"iou": 0.0, "dice": 0.0, "acc": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0}
class_iou = {"individual_tree": 0.0, "group_of_trees": 0.0}
n = 0
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(predictor.device)
        masks = masks.to(predictor.device)
        logits = predictor.model(imgs)
        m = compute_metrics_multiclass(logits, masks, num_classes=3)
        totals["iou"] += float(m.get("iou", m.get("mean_iou", 0.0)))
        totals["dice"] += float(m.get("dice", 0.0))
        totals["acc"] += float(m.get("acc", 0.0))
        totals["precision"] += float(m.get("precision", 0.0))
        totals["recall"] += float(m.get("recall", 0.0))
        totals["f1"] += float(m.get("f1_score", m.get("f1", 0.0)))
        class_iou["individual_tree"] += float(m.get("iou_individual_tree", 0.0))
        class_iou["group_of_trees"] += float(m.get("iou_group_of_trees", 0.0))
        n += 1

for k in totals:
    totals[k] /= max(n, 1)
for k in class_iou:
    class_iou[k] /= max(n, 1)

p("Val IoU", f"{totals['iou']:.4f}")
p("Val Dice", f"{totals['dice']:.4f}")
p("Val Acc", f"{totals['acc']:.4f}")
p("Val Precision", f"{totals['precision']:.4f}")
p("Val Recall", f"{totals['recall']:.4f}")
p("Val F1", f"{totals['f1']:.4f}")
p("IoU individual_tree", f"{class_iou['individual_tree']:.4f}")
p("IoU group_of_trees", f"{class_iou['group_of_trees']:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
names = list(totals.keys()) + list(class_iou.keys())
vals = [totals[k] for k in totals] + [class_iou[k] for k in class_iou]
ax.bar(names, vals)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Validation metrics (held-out split)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

t("Prediction vs GT samples")
import cv2

for entry in val_entries[:3]:
    img = load_image(train_dir, entry)
    gt = color_mask(config, entry)
    tf = get_val_augmentations(config.train.image_size)
    aug = tf(image=img, mask=np.zeros(img.shape[:2], dtype=np.uint8))
    tensor = aug["image"]
    with torch.no_grad():
        logits = predictor.model(tensor.unsqueeze(0).to(predictor.device))
        pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy().astype(np.uint8)
    if pred.shape[:2] != img.shape[:2]:
        pred = cv2.resize(pred, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    inst = separate_instances(pred)
    pred_rgb = np.zeros_like(img)
    pred_rgb[pred == 1] = config.MASK_COLORS.get("individual_tree", [0, 255, 0])
    pred_rgb[pred == 2] = config.MASK_COLORS.get("group_of_trees", [255, 0, 0])
    inst_rgb = np.zeros_like(img)
    inst_rgb[inst == 1] = config.MASK_COLORS.get("individual_tree", [0, 255, 0])
    inst_rgb[inst == 2] = config.MASK_COLORS.get("group_of_trees", [255, 0, 0])
    show_side_by_side(
        img,
        gt,
        pred_rgb,
        inst_rgb,
        titles=(entry.image_path.name, "GT", "Pred", "Instances"),
    )
    p("Polygons from instances", len(mask_to_polygons_multiclass(inst)))


## Evaluation inference + submission JSON


In [ ]:
eval_dir = config.paths.eval_images
if not eval_dir or not Path(eval_dir).exists():
    raise FileNotFoundError(f"Missing eval images: {eval_dir}")

t("Inference on evaluation set")
results = predictor.run_on_folder(eval_dir)

# Preview a few eval predictions
for r in results[:2]:
    mask = r["mask"]
    inst = separate_instances(mask)
    pred_rgb = np.zeros_like(r["image"])
    pred_rgb[mask == 1] = [0, 255, 0]
    pred_rgb[mask == 2] = [255, 0, 0]
    show_side_by_side(
        r["image"],
        pred_rgb,
        r["overlay"],
        titles=(r["name"], "Pred classes", "Overlay"),
    )
    p("Polygons", len(mask_to_polygons_multiclass(inst)))

out_path = version_folder / "submission.json"
export_submission(results, out_path, config)
p("Submission", out_path, color1=c.GREEN)
